In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [2]:
# =========================================
# 🚀 FINAL XGB PIPELINE (BALANCED ACCURACY)
# =========================================

import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

import xgboost as xgb

print("🚀 START")

# =========================================
# LOAD
# =========================================
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

target = 'Irrigation_Need'

X = train.drop(columns=[target, 'id'])
y = train[target]

X_test = test.drop(columns=['id'])
test_ids = test['id']

# =========================================
# TARGET ENCODE
# =========================================
le = LabelEncoder()
y = le.fit_transform(y)

# =========================================
# FEATURE ENGINEERING (KEEP SIMPLE BASE)
# =========================================
def fe(df):
    df['ET0'] = df['Temperature_C'] * df['Wind_Speed_kmh'] * df['Sunlight_Hours'] / (df['Humidity'] + 1)
    df['water_balance'] = df['Rainfall_mm'] + df['Previous_Irrigation_mm'] - df['ET0']
    df['stress'] = df['Temperature_C'] * (100 - df['Humidity']) / (df['Soil_Moisture'] + 1)
    df['retention'] = df['Soil_Moisture'] * df['Organic_Carbon']
    df['crop_stage'] = df['Crop_Type'] + "_" + df['Crop_Growth_Stage']
    df['region_season'] = df['Region'] + "_" + df['Season']
    return df

X = fe(X)
X_test = fe(X_test)

# =========================================
# ENCODING (XGB SAFE)
# =========================================
cat_cols = X.select_dtypes(include=['object', 'category']).columns

for col in cat_cols:
    combined = pd.concat([X[col], X_test[col]])
    codes = combined.astype('category').cat.codes.astype('int32')
    X[col] = codes[:len(X)].values
    X_test[col] = codes[len(X):].values

# =========================================
# MEMORY OPT
# =========================================
for col in X.columns:
    if X[col].dtype == 'float64':
        X[col] = X[col].astype('float32')
        X_test[col] = X_test[col].astype('float32')

print("✅ Data ready")

# =========================================
# CV
# =========================================
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# =========================================
# OBJECTIVE FUNCTION (BALANCED ACCURACY)
# =========================================
def objective(trial):

    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
    }

    scores = []

    for tr, val in skf.split(X, y):

        model = xgb.XGBClassifier(
            **params,
            n_estimators=400,
            early_stopping_rounds=50,
            eval_metric='mlogloss',
            tree_method='hist',
            n_jobs=-1
        )

        model.fit(
            X.iloc[tr], y[tr],
            eval_set=[(X.iloc[val], y[val])],
            verbose=False
        )

        preds = model.predict(X.iloc[val])
        score = balanced_accuracy_score(y[val], preds)

        scores.append(score)

    return np.mean(scores)

# =========================================
# RUN OPTUNA
# =========================================
print("🔍 OPTUNA START")

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print("✅ BEST PARAMS:", study.best_params)
print("✅ BEST SCORE:", study.best_value)

best_params = study.best_params

# =========================================
# FINAL MODEL TRAIN (FULL DATA)
# =========================================
print("🚀 FINAL TRAIN")

final_model = xgb.XGBClassifier(
    **best_params,
    n_estimators=800,
    eval_metric='mlogloss',
    tree_method='hist',
    n_jobs=-1
)

final_model.fit(X, y)

# =========================================
# PREDICTION
# =========================================
preds = final_model.predict(X_test)
preds = le.inverse_transform(preds)

# =========================================
# SUBMISSION
# =========================================
sub = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": preds
})

sub.to_csv("submission.csv", index=False)

print("🎉 DONE")

🚀 START


[I 2026-04-22 10:17:57,627] A new study created in memory with name: no-name-d83f4788-e682-4a62-8561-f76b8964642d


✅ Data ready
🔍 OPTUNA START


[I 2026-04-22 10:20:00,196] Trial 0 finished with value: 0.9616561584673885 and parameters: {'learning_rate': 0.10278752373023899, 'max_depth': 8, 'min_child_weight': 9, 'subsample': 0.9292359363539614, 'colsample_bytree': 0.9963800736435697, 'gamma': 2.2777398638667075, 'reg_alpha': 0.6085246297998825, 'reg_lambda': 0.8392062376207726}. Best is trial 0 with value: 0.9616561584673885.
[I 2026-04-22 10:23:07,079] Trial 1 finished with value: 0.9619503046697608 and parameters: {'learning_rate': 0.12104877768089373, 'max_depth': 7, 'min_child_weight': 8, 'subsample': 0.7737237462863331, 'colsample_bytree': 0.8845718792836124, 'gamma': 0.5356134462981521, 'reg_alpha': 4.404513274359824, 'reg_lambda': 1.4170617669302932}. Best is trial 1 with value: 0.9619503046697608.
[I 2026-04-22 10:25:25,126] Trial 2 finished with value: 0.9609433874064446 and parameters: {'learning_rate': 0.12057993994091983, 'max_depth': 4, 'min_child_weight': 8, 'subsample': 0.8826271106197936, 'colsample_bytree': 0.

✅ BEST PARAMS: {'learning_rate': 0.13422203872436395, 'max_depth': 6, 'min_child_weight': 8, 'subsample': 0.8660296859820502, 'colsample_bytree': 0.7754383711222106, 'gamma': 0.3574908090437811, 'reg_alpha': 4.137371041459858, 'reg_lambda': 4.120255467920414}
✅ BEST SCORE: 0.9622054553072541
🚀 FINAL TRAIN
🎉 DONE
